In [1]:
!pip install fastapi uvicorn pyngrok nest_asyncio diffusers transformers accelerate torch --quiet

In [6]:
from pyngrok import ngrok

ngrok.set_auth_token("xxxxxxxxxxxxxxxxxxxx-Your-ngrok-token-here")

In [3]:
from fastapi import FastAPI
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from diffusers import AutoPipelineForText2Image
import torch
import uuid
import os

app = FastAPI()

# Enable CORS so frontend can call the API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("Loading AI model...")

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16
).to("cuda")

print("Model loaded successfully")

IMAGE_DIR = "generated"
os.makedirs(IMAGE_DIR, exist_ok=True)

@app.get("/")
def home():
    return {"message": "AI Image Generator Running"}

@app.get("/generate")
def generate(prompt: str):

    image = pipe(
        prompt,
        num_inference_steps=2,
        guidance_scale=0.0
    ).images[0]

    filename = str(uuid.uuid4()) + ".png"
    path = os.path.join(IMAGE_DIR, filename)

    image.save(path)

    return {"image_url": f"/download/{filename}"}

@app.get("/download/{filename}")
def download_image(filename: str):

    path = os.path.join(IMAGE_DIR, filename)

    return FileResponse(path, media_type="image/png", filename=filename)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading AI model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Model loaded successfully


In [4]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

In [5]:
from pyngrok import ngrok
ngrok.kill()

public_url = ngrok.connect(8000)
print(public_url)

INFO:     Started server process [3448]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public API URL: NgrokTunnel: "https://uncephalic-macrocytic-marylyn.ngrok-free.dev" -> "http://localhost:8000"
